In [ ]:

import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
from cuml.cluster import HDBSCAN

from selectzyme.backend.embed import gen_embedding

In [ ]:
def import_results(dataset_name: str) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    """
    Imports and loads results from a dataset downloaded from Hugging Face Hub.
    Args:
        dataset_name (str): Name of the dataset to fetch from Hugging Face Hub.
    Returns:
        tuple: A tuple containing the following:
            - pd.DataFrame: DataFrame loaded from "df.parquet".
            - np.ndarray: Reduced feature matrix loaded from "x_red_mst_slc.npz".
            - np.ndarray: Minimum spanning tree (MST) loaded from "x_red_mst_slc.npz".
            - np.ndarray: Linkage matrix loaded from "x_red_mst_slc.npz".
    """
    # Download files from Hugging Face Hub
    df_path = hf_hub_download(repo_id="davari-group/selectzyme-app-data", 
                              filename=f"{dataset_name}/df.parquet", 
                              repo_type="dataset")
    npz_path = hf_hub_download(repo_id="davari-group/selectzyme-app-data", 
                               filename=f"{dataset_name}/x_red_mst_slc.npz", 
                               repo_type="dataset")
    
    # Load data
    df = pd.read_parquet(df_path)
    adata = np.load(npz_path)
    X_red = adata["X_red"]
    mst = adata["mst"]
    linkage = adata["linkage"]

    return df, X_red, mst, linkage

In [ ]:
dataset_name = "lov"
plm = "prott5"
df, X_red, mst, linkage = import_results(dataset_name)
df = df.head(300)
X = gen_embedding(df["sequence"].tolist(), plm_model=plm)

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import pandas as pd

def knn_label_consistency_scores(X, labels, k=8, n_perm=200, seed=0):
    """
    Evaluate KNN label consistency in feature space.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        labels: Cluster labels array
        k: Number of neighbors
        n_perm: Number of permutations for null distribution
        seed: Random seed
        
    Returns:
        real: Real KNN label agreement score
        null_mean: Mean of permutation null distribution
        std: Standard deviation of null distribution
    """
    rng = np.random.default_rng(seed)
    
    # Handle missing labels
    valid_idx = labels >= 0  # Exclude noise points (-1 label in HDBSCAN)
    X_valid = X[valid_idx]
    y = labels[valid_idx]
    
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan
    
    # Build KNN in feature space
    knn = NearestNeighbors(n_neighbors=k+1).fit(X_valid)  # k+1 because first neighbor is itself
    neigh_idx = knn.kneighbors(return_distance=False)[:, 1:]  # Skip self
    
    # Real score: agreement between neighbors
    real = np.mean([(y[neigh] == y[i]).mean() for i, neigh in enumerate(neigh_idx)])
    
    # Null distribution: random permutation of labels
    null_scores = []
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        s = np.mean([(y_perm[neigh] == y_perm[i]).mean() for i, neigh in enumerate(neigh_idx)])
        null_scores.append(s)
    
    null_mean = float(np.mean(null_scores))
    std = float(np.std(null_scores, ddof=1)) if len(null_scores) > 1 else np.nan
    
    return real, null_mean, std


def compute_label_correlations(labels_dict):
    """
    Compute label agreement metrics between different clusterings.
    
    Args:
        labels_dict: Dictionary of {clustering_name: labels_array}
        
    Returns:
        DataFrame with ARI and NMI correlations
    """
    results = []
    names = list(labels_dict.keys())
    
    for i, name1 in enumerate(names):
        for name2 in names[i+1:]:
            labels1 = labels_dict[name1]
            labels2 = labels_dict[name2]
            
            # Only compare valid clusters (exclude -1 noise label)
            valid_idx = (labels1 >= 0) & (labels2 >= 0)
            
            if valid_idx.sum() > 0:
                ari = adjusted_rand_score(labels1[valid_idx], labels2[valid_idx])
                nmi = normalized_mutual_info_score(labels1[valid_idx], labels2[valid_idx])
            else:
                ari, nmi = np.nan, np.nan
            
            results.append({
                'clustering1': name1,
                'clustering2': name2,
                'ARI': ari,
                'NMI': nmi
            })
    
    return pd.DataFrame(results)


In [ ]:
# Evaluate KNN agreement stability over clustering parameters
min_samples = list(range(1, 16, 2))  # tree calculation
min_cluster_size = list(range(2, 17, 2))
mst_fix_min_cluster_size = 6

knn_grid_results = []
labels_by_params = {}
mst = {}
linkage = {}

for min_size in min_cluster_size:
    for min_samp in min_samples:
        clusterer = HDBSCAN(
            min_samples=min_samp,
            min_cluster_size=min_size,
            cluster_selection_method="leaf",
            gen_min_span_tree=True,
        )
        clusterer.fit(X)
        labels = clusterer.labels_
        labels_by_params[(min_size, min_samp)] = labels

        # extract MST for fixed min_cluster_size
        if min_size == mst_fix_min_cluster_size:
            mst[min_samp] = clusterer.minimum_spanning_tree_._mst  # .to_networkx()  # # study:cuml/python/cuml/cuml/cluster/hdbscan/hdbscan.pyx: build_minimum_spanning_tree hdbscan.mst_dst, hdbscan.mst_weights
            linkage[min_samp] = clusterer.single_linkage_tree_._linkage  # .to_networkx()
    
        # Track noise points (label == -1)
        total_points = len(labels)
        noise_count = int((labels == -1).sum())
        noise_fraction = float(noise_count) / float(total_points) if total_points > 0 else np.nan
        
        real, null_mean, std = knn_label_consistency_scores(X, labels, k=10, n_perm=200)
        knn_grid_results.append({
            "min_cluster_size": min_size,
            "min_samples": min_samp,
            "knn_agreement": real,
            "permutation_null_mean": null_mean,
            "permutation_null_std": std,
            "delta_to_null": real - null_mean if pd.notna(real) and pd.notna(null_mean) else np.nan,
            "noise_count": noise_count,
            "noise_fraction": noise_fraction,
        })
        
        print(
            f"min_cluster_size={min_size}, min_samples={min_samp}: "
            f"KNN agreement={real:.4f}, permutation null={null_mean:.4f} ± {std:.4f}, "
            f"noise={noise_count}/{total_points} ({noise_fraction:.3f})"
        )

knn_grid_df = pd.DataFrame(knn_grid_results)
knn_grid_df = knn_grid_df.sort_values(["min_samples", "min_cluster_size"]).reset_index(drop=True)
print("\nKNN agreement grid:")
print(knn_grid_df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pivot results into matrices for plotting
real_pivot = knn_grid_df.pivot(index="min_samples", columns="min_cluster_size", values="knn_agreement").reindex(index=min_samples, columns=min_cluster_size)
delta_pivot = knn_grid_df.pivot(index="min_samples", columns="min_cluster_size", values="delta_to_null").reindex(index=min_samples, columns=min_cluster_size)
null_std_pivot = knn_grid_df.pivot(index="min_samples", columns="min_cluster_size", values="permutation_null_std").reindex(index=min_samples, columns=min_cluster_size)
noise_frac_pivot = knn_grid_df.pivot(index="min_samples", columns="min_cluster_size", values="noise_fraction").reindex(index=min_samples, columns=min_cluster_size)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), constrained_layout=True)
plots = [
    (real_pivot, "KNN agreement", "agreement"),
    (delta_pivot, "Agreement minus permutation baseline", "agreement - null mean"),
    (null_std_pivot, "Permutation baseline std", "std"),
    (noise_frac_pivot, "Noise fraction", "fraction of points labeled -1"),
]

for ax, (pivot, title, cbar_label) in zip(axes, plots):
    data = pivot.to_numpy(dtype=float)
    masked = np.ma.masked_invalid(data)
    im = ax.imshow(masked, origin="lower", aspect="auto", cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("min_cluster_size")
    ax.set_ylabel("min_samples")
    
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data[i, j]
            if np.isfinite(value):
                if cbar_label == "fraction of points labeled -1":
                    ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="white", fontsize=9)
                else:
                    ax.text(j, i, f"{value:.3f}", ha="center", va="center", color="white", fontsize=9)
    
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)

fig.savefig(f"lov_eval_cluster_{plm}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"lov_eval_cluster_{plm}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
label_correlations_df = compute_label_correlations(labels_by_params)
print(label_correlations_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.sparse.csgraph import shortest_path
from scipy.stats import pearsonr, spearmanr
from scipy.cluster.hierarchy import cophenet


def _mst_edges(mst_array):
    """Extract undirected edge set from a cuML MST array."""
    mst_array = np.asarray(mst_array)
    if mst_array.ndim != 2 or mst_array.shape[1] < 2:
        raise ValueError("Expected MST array with shape (n_edges, >=2).")
    edges = set()
    for row in mst_array:
        u = int(row[0])
        v = int(row[1])
        if u == v:
            continue
        edges.add(tuple(sorted((u, v))))
    return edges


def pairwise_mst_jaccard(mst_dict):
    """Compute pairwise Jaccard similarity between MST edge sets."""
    items = list(mst_dict.items())
    results = []
    for i, (ms1, mst1) in enumerate(items):
        edges1 = _mst_edges(mst1)
        for ms2, mst2 in items[i + 1:]:
            edges2 = _mst_edges(mst2)
            union = edges1 | edges2
            inter = edges1 & edges2
            score = len(inter) / len(union) if union else np.nan
            results.append({
                "min_samples_1": ms1,
                "min_samples_2": ms2,
                "mst_jaccard": score,
                "shared_edges": len(inter),
                "union_edges": len(union),
            })
    return pd.DataFrame(results)


def pairwise_shortest_path_correlation(mst_dict, X):
    """Compare MSTs by correlation of all-pairs shortest-path distances."""
    items = list(mst_dict.items())
    results = []
    for i, (ms1, mst1) in enumerate(items):
        edges1 = np.asarray(mst1)
        n_nodes = X.shape[0]
        if edges1.shape[1] >= 3:
            dist1 = np.zeros((n_nodes, n_nodes), dtype=float)
            for u, v, w in edges1[:, :3]:
                u = int(u)
                v = int(v)
                dist1[u, v] = float(w)
                dist1[v, u] = float(w)
        else:
            dist1 = np.zeros((n_nodes, n_nodes), dtype=float)
            for u, v in edges1[:, :2].astype(int):
                dist1[u, v] = 1.0
                dist1[v, u] = 1.0
        sp1 = shortest_path(dist1, directed=False, unweighted=False)
        vec1 = sp1[np.triu_indices_from(sp1, k=1)]

        for ms2, mst2 in items[i + 1:]:
            edges2 = np.asarray(mst2)
            if edges2.shape[1] >= 3:
                dist2 = np.zeros((n_nodes, n_nodes), dtype=float)
                for u, v, w in edges2[:, :3]:
                    u = int(u)
                    v = int(v)
                    dist2[u, v] = float(w)
                    dist2[v, u] = float(w)
            else:
                dist2 = np.zeros((n_nodes, n_nodes), dtype=float)
                for u, v in edges2[:, :2].astype(int):
                    dist2[u, v] = 1.0
                    dist2[v, u] = 1.0
            sp2 = shortest_path(dist2, directed=False, unweighted=False)
            vec2 = sp2[np.triu_indices_from(sp2, k=1)]

            valid = np.isfinite(vec1) & np.isfinite(vec2)
            if valid.sum() > 1:
                pearson = pearsonr(vec1[valid], vec2[valid]).statistic
                spearman = spearmanr(vec1[valid], vec2[valid]).correlation
            else:
                pearson, spearman = np.nan, np.nan

            results.append({
                "min_samples_1": ms1,
                "min_samples_2": ms2,
                "shortest_path_pearson": pearson,
                "shortest_path_spearman": spearman,
            })
    return pd.DataFrame(results)


def pairwise_linkage_cophenetic_correlation(linkage_dict):
    """Compute pairwise cophenetic distance correlation between linkage matrices."""
    items = list(linkage_dict.items())
    results = []
    for i, (ms1, link1) in enumerate(items):
        coph1 = np.asarray(cophenet(np.asarray(link1)))
        for ms2, link2 in items[i + 1:]:
            coph2 = np.asarray(cophenet(np.asarray(link2)))
            valid = np.isfinite(coph1) & np.isfinite(coph2)
            if valid.sum() > 1:
                pearson = pearsonr(coph1[valid], coph2[valid]).statistic
                spearman = spearmanr(coph1[valid], coph2[valid]).correlation
            else:
                pearson, spearman = np.nan, np.nan
            results.append({
                "min_samples_1": ms1,
                "min_samples_2": ms2,
                "cophenetic_pearson": pearson,
                "cophenetic_spearman": spearman,
            })
    return pd.DataFrame(results)

mst_jaccard_df = pairwise_mst_jaccard(mst)
shortest_path_corr_df = pairwise_shortest_path_correlation(mst, X)
linkage_cophenetic_corr_df = pairwise_linkage_cophenetic_correlation(linkage)

print("Pairwise MST Jaccard similarity:")
print(mst_jaccard_df)
print("\nPairwise shortest-path distance correlation:")
print(shortest_path_corr_df)
print("\nPairwise linkage cophenetic correlation:")
print(linkage_cophenetic_corr_df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def pairwise_metric_to_square(df, value_col):
    min_samples_vals = sorted(set(df["min_samples_1"]).union(df["min_samples_2"]))
    mat = pd.DataFrame(np.nan, index=min_samples_vals, columns=min_samples_vals, dtype=float)

    for _, row in df.iterrows():
        i = int(row["min_samples_1"])
        j = int(row["min_samples_2"])
        val = row[value_col]
        mat.loc[i, j] = val
        mat.loc[j, i] = val

    np.fill_diagonal(mat.values, 1.0 if value_col == "mst_jaccard" else np.nan)
    return mat

heatmap_specs = [
    ("mst_jaccard", "MST Jaccard"),
    ("shared_edges", "Shared MST edges"),
    ("union_edges", "Union MST edges"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

for ax, (metric, title) in zip(axes, heatmap_specs):
    metric_mat = pairwise_metric_to_square(mst_jaccard_df, metric)
    data = metric_mat.to_numpy(dtype=float)
    masked = np.ma.masked_invalid(data)

    cmap = "Blues"
    im = ax.imshow(masked, origin="lower", aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(range(len(metric_mat.columns)))
    ax.set_xticklabels(metric_mat.columns)
    ax.set_yticks(range(len(metric_mat.index)))
    ax.set_yticklabels(metric_mat.index)
    ax.set_xlabel("min_samples")
    ax.set_ylabel("min_samples")

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            if np.isfinite(val):
                txt = f"{val:.2f}" if metric == "mst_jaccard" else f"{int(val)}"
                ax.text(j, i, txt, ha="center", va="center", color="white", fontsize=8)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metric)

fig.savefig(f"lov_eval_mst_{plm}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"lov_eval_mst_{plm}.pdf", bbox_inches="tight")
plt.show()